# NISAR workflow (NASA Earthdata)
Same steps as `Biomass_Downloader.ipynb`, for the NISAR L-band products that NASA distributes through the Alaska Satellite Facility. The logic lives in the `biomass` package (`src/biomass`). Install once with `pip install -e ".[geo,dev]"`, edit only the configuration cell and run top to bottom.

It needs a NASA Earthdata Login token in `nasa_token.txt`, next to `token.txt` (https://urs.earthdata.nasa.gov > Generate Token; it lasts 60 days).

In [ ]:
from pathlib import Path
from biomass import (NisarClient, SearchQuery, Downloader, nisar_items_table, assets_table,
                     footprint_map, inspect_gcov, plot_gcov, gcov_map)

# --- Configuration ---
QUERY = SearchQuery(
    collection="NISAR_L2_GCOV_PROVISIONAL_V1_1",   # the collection is the product type (see Explore)
    date_start="2025-10-28",                       # first provisional products
    date_end="2026-07-01",
    bbox=[-81.00, -20.00, -35.00, 10.00],          # [min_lon, min_lat, max_lon, max_lat]
    aoi_file=None,                                 # e.g. "my_area.geojson" (overrides bbox)
    max_items=10,                                  # results come oldest first
)
# Acquisition mode of the main band (frequency A); None accepts any
POLARISATION = "DH"                                # "SH"/"SV" one channel, "DH" HH+HV, "DV" VV+VH, "QP" all four;
                                                   # or a list, e.g. ["DH", "QP"] for everything with HH and HV
BANDWIDTH = None                                   # MHz: 5, 20, 40 or 77 (more bandwidth = finer range resolution)
BROWSE_ASSETS = ["browse_latlon"]                  # small preview images
PRODUCT_ASSETS = ["product"]                       # the .h5 file: about 5 GB each
OUTPUT_DIR = Path("../data/nisar")

client = NisarClient()
client.auth.access_token(); print("Token OK")

## Explore

In [ ]:
client.collections()

## Search

In [ ]:
items = client.search(QUERY, polarisation=POLARISATION, bandwidth=BANDWIDTH)
nisar_items_table(items)

In [ ]:
footprint_map(items, QUERY)

## Download

In [ ]:
assets_table(items[0])

In [ ]:
downloader = Downloader(client.auth, OUTPUT_DIR, item_folders=False)
browse = downloader.download(items, BROWSE_ASSETS)

In [ ]:
files = downloader.download(items[:1], PRODUCT_ASSETS)   # about 5 GB per product: start with one

## Inspect

In [ ]:
inspect_gcov(files[0])

## Visualize
`plot_gcov` shows every backscatter term of one file (gamma0 in dB, on its UTM grid). `gcov_map` lays the downloaded files over a map: `band="rgb"` is a colour composite (R=HHHH, G=HVHV, B=VVVV, or HHHH/HVHV when there is no VV), or pass `"HHHH"`, `"HVHV"`... for a single term. Both read the coarsest grid in the file by default; pass `frequency="A"` for the main 10 m one.

In [ ]:
h5_files = sorted(OUTPUT_DIR.glob("*.h5"))
plot_gcov(h5_files[0]);

In [ ]:
gcov_map(h5_files)   # one layer per file; toggle them with the control in the top-right corner